In [28]:
# %%writefile Ind_OBV_EX.py

import numpy as np
import pandas as pd

import QUANTAXIS as QA

import Ind_Model_Base

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import Analysis_Funs as af
import Sample_Tools as smpl
import Pretreat_Tools as pretreat


%load_ext autoreload
%autoreload 2
%aimport Analysis_Funs,Sample_Tools,Pretreat_Tools

class OBV(Ind_Model_Base.Ind_Model):
    '''能量潮指标改进版'''
    def __init__(self,data, frequence=QA.FREQUENCE.DAY):
        super().__init__(data, 'OBV_EX', frequence)
    
    def on_set_params_default(self):
        return {'SHORT':5, 'LONG':15}
        
    def on_indicator_structuring(self, data):
        return self.excute_for_multicode(data, self.kernel, **self.pramas)

    
    def on_desition_structuring(self, data, ind_data):
        """
        1.短期量价穿越长期,res为1，买入信号参考。
        2.相反则res为-1，卖出信号参考。
        """
        return pd.DataFrame({'res':ind_data['CROSS_JC'] + ind_data['CROSS_SC']*-1})
        
    def kernel(self,dataframe,SHORT=5,LONG=15):
        '''多空比率净额= [（收盘价－最低价）－（最高价-收盘价）] ÷（ 最高价－最低价）×V'''
        long_short_ratio=((dataframe.close - dataframe.low) - (dataframe.high - dataframe.close)) / (dataframe.high-dataframe.low) * dataframe.volume
        
        short =QA.EMA(long_short_ratio,SHORT)
        long = QA.EMA(long_short_ratio,LONG)
        

        CROSS_JC=QA.CROSS(short, long)
        CROSS_SC=QA.CROSS(long, short)

        return pd.DataFrame({'LSR':long_short_ratio,'CROSS_JC':CROSS_JC, 'CROSS_SC':CROSS_SC})
#         return pd.DataFrame({'LSR':long_short_ratio})
    
    
    def plot(self,figsize=(1120/72,420/72)) -> dict:
        fig = plt.figure(figsize=figsize)
        groups = self.ind_df.groupby(level=1)
        for idx,item in enumerate(groups):
            inds_ = item[1].reset_index('code',drop=True)
            ax = fig.add_subplot(len(groups),1,idx+1)
             
            inds_.plot(ax=ax,legend=True)
            ax.set_title(item[0],color='r', loc ='left', pad=-10)
            ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
            plt.xticks(rotation = 0)
            
    
    def plot_mix(self,figsize=(1120/72,420/72)) -> dict:
        fig = plt.figure(figsize=figsize)
        groups = self.ind_df.groupby(level=1)
        def x1(item):
            inds_ = item.reset_index('code',drop=True)
            plt.plot(inds_['LSR'])
#             print(item.name)
        groups.apply(x1)
        plt.legend(groups.groups.keys())
        
    def self_test(self):
        data = get_sample_by_zs(name='上证50', end='2020-06-29', gap=250, only_main=True)
        df = resample_stockdata_low(data.data,freq="M")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
# data = smpl.get_sample_by_zs(name='上证50', end='2020-06-29', gap=250, only_main=True)
# df = smpl.resample_stockdata_low(data.data,freq="M")
# ret_forward = smpl.get_forward_return(df,'close')


# obv = OBV(data.data)
# obv.fit()
# ind = obv.ind_df['LSR']
# factor_standardized = pretreat.standardize(ind, multi_code=True)
# rank_ic = af.get_rank_ic(factor_standardized, ret_forward)
# rank_ic
# af.get_ic_desc(rank_ic)
# af.get_ic_ir(rank_ic)
# af.get_winning_rate(rank_ic)

QA.QA_fetch_financial_report_adv('000002','2019-01-01','2021-05-01').data


,,code,report_date,EPS,deductEPS,undistributedProfitPerShare,netAssetsPerShare,capitalReservePerShare,ROE,operatingCashFlowPerShare,moneyFunds,...,Net increase in funds transferred in,Net increase in funds from repo business,Net increase in loans and advances to customers,Net increase in deposits with central banks and interbank,Cash paid for claims on original insurance contracts,"Cash paid for interest, fees and commissions",Cash paid for policy dividends,cash received from minority shareholders investment in subsidiaries,Dividends and profits paid by subsidiaries to minority shareholders,Depreciation and amortization of investment properties
report_date,code,,,,,,,,,,,,,,,,,,,,,
2019-03-31,000002,000002,2019-03-31,0.102,0.102222,8.4106,14.272100,0.7213,0.711,-2.420,1.432204e+11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45049.320312,3.021064e+05,0.0
2019-06-30,000002,000002,2019-06-30,1.060,1.060000,8.1183,14.246600,1.0795,7.354,0.783,1.438688e+11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,211234.359375,5.844564e+05,0.0
2019-09-30,000002,000002,2019-09-30,1.631,1.608152,8.6845,14.800200,1.0557,10.905,0.154,1.072402e+11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,205571.921875,6.556104e+05,0.0
2019-12-31,000002,000002,2019-12-31,3.470,3.420000,8.4366,16.639200,1.0958,20.670,4.042,1.661946e+11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,517088.250000,1.114748e+06,0.0
2020-03-31,000002,000002,2020-03-31,0.111,0.090001,8.5472,16.692801,1.0918,0.662,-0.261,1.732716e+11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91743.632812,9.307986e+04,0.0
